In [ ]:
import os, time, math, datetime as dt
import dotenv
import duckdb
import geopandas as gpd
import pandas as pd
import numpy as np
import shapely
from shapely import force_2d
from shapely.prepared import prep
from shapely.geometry import LineString, MultiLineString
from shapely.ops import polygonize
from shapely.geometry import box
from shapely.validation import make_valid

from shapely.strtree import STRtree
from shapely import union_all

import networkx as nx
import logging
from collections import deque, defaultdict
import gc 
from concurrent.futures import ProcessPoolExecutor, as_completed
from shapely.strtree import STRtree
from time import perf_counter
import pyarrow.parquet as pq

gc.enable()

dotenv.load_dotenv()


wrk_dir=os.getenv("WORKING_DIR")
brwmb_targets_csv=os.getenv("BRWMB_TARGETS")
source_data=os.getenv("source_data")
output_parquets=os.getenv("OUTPUT_PARQUETS")
marxan_inputs=os.getenv("MARXAN_INPUTS")
input_parquets=os.getenv("INPUT_PARQUETS")
output_dir = os.path.join(wrk_dir, "brwmb_parquets")
mineral_licks_gdb=os.path.join(marxan_inputs,'MineralLicks_Hex_2024_12_16','MineralLicks_Hex_2024_12_16.gdb')
licks_layer='MineralLicks_2024_12_16'
grid_path=os.path.join(input_parquets,'hexagons_3ha.parquet')
grid_id_col='GRID_ID'
cia1_path=os.path.join(input_parquets,'cia_1.parquet')
cia2_path=os.path.join(input_parquets,'cia_2.parquet')
weighted_surface_path = os.path.join(output_dir, "step_3f_weighted_surface.parquet")
scenario5_setup_gdb=os.path.join(source_data,'Scenario5_Setup.gdb')
wmb_layer='priority_WMB'
rec_for_path=os.path.join(output_dir,'step_3c_Recruitment_Forest.parquet')
aflb_parquet=os.path.join(input_parquets,'thlb_select.parquet')
step2_lyr=os.path.join(output_parquets,'ret_class_2e_all.parquet')
target_crs = "EPSG:3005"
wmv_inputs=os.getenv("WMV_INPUTS")
ret_class=os.path.join(input_parquets,'rec_class_all.parquet')
table_outputs=os.getenv('TABLE_OUPUTS')

patche_area_3a=10

max_rings = 2
max_bridging= 2
#set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
info=logging.info
debug=logging.debug
error=logging.error

In [ ]:
#variables 
if not os.path.exists(output_dir):
    os.makedirs(output_dir)


REC_CAT = "Rec_Cat"
REC_SHORT = "Rec_Cat_short"
CRS_EXPECTED = 3005

conn = duckdb.connect()
conn.install_extension("httpfs")
conn.install_extension("spatial")
conn.install_extension("arrow")
conn.load_extension("spatial")
conn.load_extension("httpfs")
conn.load_extension("arrow")


In [ ]:
def return_gdf(sql):
    df = conn.sql(sql).to_df()
    df['geometry'] = gpd.GeoSeries.from_wkt(df['wkt'])
    df = gpd.GeoDataFrame(df).set_crs(3005, allow_override=True)
    df = df.drop(columns=['wkt'])
    info(f" length of df: {len(df)}")
    return df


def to3005(g):
    if g is None or (hasattr(g, "empty") and g.empty):
        return g
    return g if g.crs == target_crs else g.to_crs(target_crs)


def fix(gdf):
    if gdf is None or gdf.empty:
        return gdf
    try:
        gdf["geometry"] = gdf.geometry.apply(lambda g: make_valid(g) if g and not g.is_valid else g)
    except Exception:
        gdf["geometry"] = gdf.buffer(0)
    return gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty].copy()


def intersect_ids(grid_gdf, poly_gdf,grid_id_col):
    if poly_gdf is None or poly_gdf.empty:
        return set()
    j = gpd.sjoin(grid_gdf[[grid_id_col, "geometry"]], poly_gdf[["geometry"]], predicate="intersects", how="inner")
    return set(j[grid_id_col].unique())

def _poly_only(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    if gdf is None or gdf.empty:
        return gdf
    g = gdf[gdf.geometry.geom_type.isin(["Polygon","MultiPolygon"])].copy()
    g = g[g.geometry.notna() & ~g.geometry.is_empty]
    return g

def ids_by_coverage(grid_gdf, poly_gdf, grid_id_col, min_prop=0.10):
    """Per-hex coverage proportion with robust fallbacks."""
    if poly_gdf is None or poly_gdf.empty:
        return set(), pd.DataFrame(index=[], columns=["prop"])
    A = (grid_gdf[[grid_id_col, "geometry"]]
         .drop_duplicates(subset=[grid_id_col])
         .copy())
    B = _poly_only(poly_gdf)
    if A.empty or B.empty:
        return set(), pd.DataFrame(index=[], columns=["prop"])
    try:
        inter = gpd.overlay(A[[grid_id_col, "geometry"]], B[["geometry"]], how="intersection")
    except NotImplementedError:
        # Safe fallback: sjoin candidates + pairwise intersection
        cand = gpd.sjoin(
            A[[grid_id_col, "geometry"]].reset_index().rename(columns={"index": "_iA"}),
            B[["geometry"]],
            predicate="intersects",
            how="inner"
        )
        if cand.empty:
            return set(), pd.DataFrame(index=[], columns=["prop"])
        left  = A.iloc[cand["_iA"].to_numpy()].reset_index(drop=True)
        right = B.iloc[cand["index_right"].to_numpy()].reset_index(drop=True)
        geoms, ids = [], []
        for i in range(len(left)):
            g = left.geometry.iloc[i].intersection(right.geometry.iloc[i])
            if not g.is_empty:
                geoms.append(g)
                ids.append(left[grid_id_col].iloc[i])
        if not geoms:
            return set(), pd.DataFrame(index=[], columns=["prop"])
        inter = gpd.GeoDataFrame({grid_id_col: ids}, geometry=geoms, crs=A.crs)

    if inter.empty:
        return set(), pd.DataFrame(index=[], columns=["prop"])

    inter["part_area"] = inter.geometry.area
    hex_area = A.set_index(grid_id_col).geometry.area.to_dict()
    inter["hex_area"] = inter[grid_id_col].map(hex_area)
    inter = inter[inter["hex_area"] > 0]
    if inter.empty:
        return set(), pd.DataFrame(index=[], columns=["prop"])

    inter["prop"] = inter["part_area"] / inter["hex_area"]
    props = inter.groupby(grid_id_col, as_index=True)["prop"].sum().to_frame()
    props["prop"] = props["prop"].clip(upper=1.0)
    ids = set(props.index[props["prop"] >= min_prop].tolist())
    return ids, props


def ids_by_intersects(grid_gdf, poly_gdf, grid_id_col):
    """Loose membership by intersects, unique GRID_IDs."""
    if poly_gdf is None or poly_gdf.empty:
        return set()
    j = gpd.sjoin(
        grid_gdf[[grid_id_col, "geometry"]].drop_duplicates(subset=[grid_id_col]),
        poly_gdf[["geometry"]],
        predicate="intersects",
        how="inner"
    )
    return set(j[grid_id_col].unique())

def union_geom(gdf):
    if gdf is None or gdf.empty:
        return None
    u = shapely.union_all(gdf.geometry.values)
    return None if (u is None or shapely.is_empty(u)) else u


def set_grid_precision(gdf, grid=0.5):
    # Optional: snap vertices to a 0.5 m grid to reduce complexity & overlay issues
    if gdf is None or gdf.empty: return gdf
    gdf["geometry"] = shapely.set_precision(gdf.geometry.values, grid)
    return gdf

def dissolve_single(gdf):
    if gdf is None or gdf.empty: return gdf
    u = shapely.union_all(gdf.geometry.values)
    if u is None or shapely.is_empty(u):
        return gpd.GeoDataFrame(geometry=[], crs=gdf.crs)
    return gpd.GeoDataFrame(geometry=[u], crs=gdf.crs)




In [ ]:


# def build_components_within_class(
#     g: gpd.GeoDataFrame,
#     touches_only: bool = False,
# ) -> pd.Series:
#     """
#     Assign a connected-component id within each (Rec_Cat, Rec_Cat_short) group.
#     Returns Series aligned to g.index with dtype Int64.
#     """
#     comp_id = pd.Series(index=g.index, dtype="Int64")

#     for (rc, rs), sub in g.groupby([REC_CAT, REC_SHORT], sort=False):
#         if sub.empty:
#             continue

#         sindex = sub.sindex
#         # Prepared only used when we don't pass a predicate
#         prepared = {idx: prep(geom) for idx, geom in sub.geometry.items()}

#         visited: set[int] = set()
#         comp = 0
#         idxs = sub.index.to_list()

#         for start in idxs:
#             if start in visited:
#                 continue

#             stack = [start]
#             visited.add(start)
#             members = []

#             while stack:
#                 u = stack.pop()
#                 members.append(u)
#                 gu = sub.at[u, "geometry"]
#                 if gu is None or gu.is_empty:
#                     continue

#                 # Use geometry, not bounds, with the spatial index
#                 if touches_only:
#                     cand_pos = sindex.query(gu, predicate="touches")
#                     # 'touches' excludes overlaps; no extra predicate needed
#                     cand_idx = sub.index.take(cand_pos)
#                     neigh_idxs = cand_idx
#                 else:
#                     # Inclusive adjacency (touch OR overlap)
#                     cand_pos = sindex.query(gu)  # envelope hits
#                     cand_idx = sub.index.take(cand_pos)
#                     # Refine with precise predicate
#                     neigh_idxs = [
#                         v for v in cand_idx
#                         if v != u
#                         and v not in visited
#                         and sub.at[v, "geometry"] is not None
#                         and not sub.at[v, "geometry"].is_empty
#                         and prepared[u].intersects(sub.at[v, "geometry"])
#                     ]

#                 for v in neigh_idxs:
#                     if v == u or v in visited:
#                         continue
#                     visited.add(v)
#                     stack.append(v)

#             # Assign component id
#             for m in members:
#                 comp_id.at[m] = comp
#             comp += 1

#     return comp_id

def _norm_classes(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    g = gdf.copy()
    # Ensure key columns exist
    for c in ["Rec_Cat", "Rec_Cat_short", "aflb_fact", "geometry"]:
        if c not in g.columns:
            raise KeyError(f"Missing column '{c}' in provided dataset")
    # Normalize types/values
    g["Rec_Cat"] = pd.to_numeric(g["Rec_Cat"], errors="coerce").astype("Int64")
    g["Rec_Cat_short"] = g["Rec_Cat_short"].astype(str).str.strip().str.upper()
    g = g.loc[g.geometry.notna() & ~g.geometry.is_empty].copy()
    return g

def _sum_intersection_area_ha(units_gdf: gpd.GeoDataFrame,
                              units_id: str,
                              layer_gdf: gpd.GeoDataFrame) -> pd.Series:
    if layer_gdf.empty:
        return pd.Series(dtype='float64', name='area_ha')
    A = units_gdf[[units_id, 'geometry']].dropna(subset=['geometry'])
    B = layer_gdf[['geometry']].dropna(subset=['geometry'])
    ix = gpd.overlay(A, B, how='intersection', keep_geom_type=True)
    if ix.empty:
        return pd.Series(dtype='float64', name='area_ha')
    ix['area_ha'] = ix.geometry.area / 10_000.0
    return ix.groupby(units_id)['area_ha'].sum().rename('area_ha')



In [ ]:

def get_single_geom(
    gdf: gpd.GeoDataFrame,
    *,
    precision_snap: float | None = 1.0,
    simplify_tolerance: float | None = None
):
    if gdf is None or gdf.empty:
        return None
    geom = gdf.geometry.iloc[0]
    if shapely.is_empty(geom):
        return None
    if precision_snap is not None:
        geom = shapely.set_precision(geom, precision_snap)
    if simplify_tolerance is not None and simplify_tolerance > 0:
        geom = shapely.simplify(geom, simplify_tolerance, preserve_topology=True)
    return geom

def make_tile_ids_for_grid(grid: gpd.GeoDataFrame, tile_m: float) -> pd.Series:
    cx = grid.geometry.centroid.x.values
    cy = grid.geometry.centroid.y.values
    minx, miny, maxx, maxy = grid.total_bounds
    ix = np.floor((cx - minx) / tile_m).astype(int)
    iy = np.floor((cy - miny) / tile_m).astype(int)
    return pd.Series(list(zip(ix, iy)), index=grid.index, dtype="object")

def intersection_overlay_by_gid(cells: gpd.GeoDataFrame, layer_geom):
    """
    Intersect subset of cells with a single geometry.
    Returns a Series: total intersection ha per cell index.
    """
    if cells.empty or layer_geom is None or shapely.is_empty(layer_geom):
        return pd.Series(dtype=float)

    work = cells.copy()
    work["__gid__"] = work.index

    layer = gpd.GeoDataFrame({"_": [1]}, geometry=[layer_geom], crs=cells.crs)
    inter = gpd.overlay(work[["__gid__", "geometry"]], layer[["geometry"]], how="intersection")
    if inter.empty:
        return pd.Series(dtype=float)

    inter["__ha__"] = inter.geometry.area / 10000.0
    return inter.groupby("__gid__")["__ha__"].sum()

def _process_tile_worker(args):
    """
    Runs in a separate process. Returns (tile_idx, delta_series, per_layer_stats)
    where delta_series is indexed by original grid index with weight additions.
    """
    (tile_idx, tile_id, tile_bounds, tile_index, grid_slice_wkb, grid_area_ha,
     layer_wkbs, weight_mode) = args

    t0 = time.time()
    minx, miny, maxx, maxy = tile_bounds
    pad = 5.0
    tile_box = box(minx - pad, miny - pad, maxx + pad, maxy + pad)

    # Rehydrate grid slice from WKB
    tile_cells = gpd.GeoDataFrame(
        geometry=[shapely.from_wkb(w) for w in grid_slice_wkb],
        index=tile_index,
        crs="EPSG:3005"
    )
    # Keep area_ha Series for this tile
    area_ha_tile = pd.Series(grid_area_ha, index=tile_index, dtype=float)

    # Accumulator for this tile
    delta = pd.Series(0.0, index=tile_cells.index, dtype=float)

    per_layer_stats = []  # (name, hits, rows, seconds)

    for name, (wkb, w) in layer_wkbs.items():
        geom = shapely.from_wkb(wkb)

        # bbox reject
        if not shapely.intersects(geom, tile_box):
            per_layer_stats.append((name, 0, 0, 0.0))
            continue

        # clip to tile
        t_layer0 = time.time()
        tile_clip = shapely.intersection(geom, tile_box)
        if shapely.is_empty(tile_clip):
            per_layer_stats.append((name, 0, 0, time.time() - t_layer0))
            continue

        # intersect with cells
        hits_series = intersection_overlay_by_gid(tile_cells, tile_clip)  # Series indexed by cell idx
        n_rows = int(hits_series.size)
        if n_rows == 0:
            per_layer_stats.append((name, 0, 0, time.time() - t_layer0))
            continue

        if weight_mode == "proportion":
            prop = (hits_series / area_ha_tile.loc[hits_series.index]).clip(upper=1.0)
            delta.loc[prop.index] += float(w) * prop
        elif weight_mode == "area":
            delta.loc[hits_series.index] += float(w) * hits_series
        else:
            raise ValueError("weight_mode must be 'proportion' or 'area'")

        per_layer_stats.append((name, int(hits_series.size), n_rows, time.time() - t_layer0))

    return tile_idx, tile_id, delta, per_layer_stats, time.time() - t0

def build_wvm_additive_cells_tiled_parallel(
    grid: gpd.GeoDataFrame,
    layers: dict[str, tuple[gpd.GeoDataFrame, float]],
    *,
    tile_km: float = 10.0,
    weight_mode: str = "proportion",           # "proportion" or "area"
    simplify_tolerance: float | None = None,
    precision_snap: float | None = 1.0,
    non_forest_mask: gpd.GeoDataFrame | None = None,
    max_workers: int | None = None,            # default: os.cpu_count()
    save_intermediate_tile_weights: bool = False,
    output_dir: str | None = None
):
    t0 = time.time()
    info("=== [3f] Parallel Tiled additive WVM start ===")

    # Prep grid
    grid = to3005(grid)
    if "area_ha" not in grid.columns:
        grid["area_ha"] = grid.geometry.area / 10000.0
    if "weight_sum" not in grid.columns:
        grid["weight_sum"] = 0.0

    if non_forest_mask is not None and not non_forest_mask.empty:
        info("[3f] Applying non-forest mask to grid via intersection…")
        grid = gpd.overlay(grid, to3005(non_forest_mask)[["geometry"]], how="intersection")
        grid["area_ha"] = grid.geometry.area / 10000.0
        info(f"[3f] Grid after mask: {len(grid):,} cells")

    # Load single geoms (already dissolved)
    info("[3f] Loading single geoms (no dissolve) and encoding as WKB …")
    layer_wkbs: dict[str, tuple[bytes, float]] = {}
    for name, (gdf, w) in layers.items():
        gdf = to3005(gdf)
        if gdf is None or gdf.empty or float(w) == 0.0:
            info(f"[3f] {name}: empty or weight=0 → skip")
            continue
        geom = get_single_geom(gdf, precision_snap=precision_snap, simplify_tolerance=simplify_tolerance)
        if geom is None or shapely.is_empty(geom):
            info(f"[3f] {name}: empty after prep → skip")
            continue
        layer_wkbs[name] = (shapely.to_wkb(geom), float(w))
        bx = shapely.bounds(geom)
        info(f"[3f] {name}: ready (w={w}) | bounds=({bx[0]:.0f},{bx[1]:.0f},{bx[2]:.0f},{bx[3]:.0f})")

    if not layer_wkbs:
        raise RuntimeError("[3f] No valid layers to process")

    # Build tiles
    tile_m = tile_km * 1000.0
    tile_ids = make_tile_ids_for_grid(grid, tile_m)
    groups = grid.groupby(tile_ids, sort=False)
    n_tiles = len(groups)
    info(f"[3f] Tiles (approx {tile_km} km): {n_tiles:,}")

    # Prepare tasks
    tasks = []
    for idx, (tile_id, tile_df) in enumerate(groups, start=1):
        minx, miny, maxx, maxy = tile_df.total_bounds
        # lightweight payload: WKB geometries + areas for this tile only
        grid_slice_wkb = [shapely.to_wkb(g) for g in tile_df.geometry.values]
        grid_area_ha = tile_df["area_ha"].to_numpy()
        tile_index = tile_df.index.to_numpy()
        tasks.append((
            idx, tile_id, (minx, miny, maxx, maxy), tile_index,
            grid_slice_wkb, grid_area_ha, layer_wkbs, weight_mode
        ))

    # Execute in parallel
    tile_durations = []
    futures = []
    results = []
    with ProcessPoolExecutor(max_workers=max_workers) as exe:
        info(f"[3f] Launching {len(tasks):,} tile tasks with max_workers={max_workers or os.cpu_count()} …")
        for task in tasks:
            futures.append(exe.submit(_process_tile_worker, task))

        for fut in as_completed(futures):
            tile_idx, tile_id, delta, per_layer_stats, dt_tile = fut.result()
            tile_durations.append(dt_tile)
            results.append((tile_idx, tile_id, delta, per_layer_stats, dt_tile))

            # Live progress per tile
            avg = float(np.mean(tile_durations))
            eta = (n_tiles - len(tile_durations)) * avg
            info(f"[3f][Tile {tile_idx}/{n_tiles}] id={tile_id} done in {dt_tile:0.1f}s | avg={avg:0.1f}s | ETA≈{eta/60:0.1f} min")

            # Optional checkpoint write-back per tile
            if save_intermediate_tile_weights and output_dir:
                # apply just this tile's delta to a copy for checkpoint
                tmp = grid.loc[delta.index, [grid.geometry.name, "weight_sum"]].copy()
                tmp.loc[delta.index, "weight_sum"] = (grid.loc[delta.index, "weight_sum"] + delta.values)
                ck_path = os.path.join(output_dir, f"step_3f_tile_{tile_idx:04d}_weights.parquet")
                tmp.to_parquet(ck_path, index=True)
                info(f"    ~ checkpoint saved: {ck_path}")

    # Merge all deltas back (sum by index)
    info("[3f] Merging tile deltas …")
    # Combine by concatenating as DataFrame and summing per index
    if results:
        all_deltas = pd.concat([r[2].rename("delta") for r in results]).groupby(level=0).sum()
        grid.loc[all_deltas.index, "weight_sum"] += all_deltas.values

    # Final metrics
    if weight_mode == "proportion":
        grid["weighted_ha"] = grid["weight_sum"] * grid["area_ha"]
    else:
        grid["weighted_ha"] = grid["weight_sum"]

    nonzero = grid.loc[grid["weight_sum"] > 0].copy()
    info(f"[3f] Cells with weight > 0: {len(nonzero):,}")

    if len(nonzero) == 0:
        raise RuntimeError("[3f] No cells accumulated any weight — check inputs")

    thr = nonzero["weighted_ha"].quantile(0.75)
    optimal_cells = nonzero.loc[nonzero["weighted_ha"] >= thr].copy()
    info(f"[3f] Top 25% threshold = {thr:0.6f} | Optimal cells = {len(optimal_cells):,}")


    return grid, optimal_cells, thr

In [ ]:
#3a a)	From the merged layer created in 2e), select all stands >140. Create layer from selection. Merge adjoining stands to form patches (smallest patch unit= 2 adjoining stands).
# Calculate the mean shape index (MSI) of each patch. Select patches with MSI <1 and an area >20ha. Create layer from selection
sql_3a = f"""
SELECT *,ST_AsText(geometry) as wkt, from '{step2_lyr}'
    WHERE SIFA_2 >=140
"""
old_stands_gdf = return_gdf(sql_3a)
info(f"Old stands gdf has {len(old_stands_gdf)} features")

if "FEATURE_ID" in old_stands_gdf.columns:
    old_stands_gdf = old_stands_gdf.rename(columns={"FEATURE_ID": "orig_id"})
else:
    old_stands_gdf = old_stands_gdf.reset_index(names="orig_id")
old_stands_gdf["__row_n__"] = np.arange(len(old_stands_gdf), dtype=np.int64)

exploded = old_stands_gdf.explode(index_parts=False, ignore_index=False)
exploded["part_id"] = (
    exploded.groupby("orig_id", dropna=False)["__row_n__"]
            .rank(method="first")
            .astype(int) - 1
)
exploded.to_parquet(os.path.join(output_dir, "step_3a_old_stands_gdf.parquet"))

info("[3a] Building touch-adjacent patches from exploded stands")
t0 = perf_counter()

gdf = exploded.reset_index(drop=False).rename(columns={'index': 'row_idx'}).copy()
gdf = gdf[['row_idx', 'orig_id', 'part_id', 'geometry']]

sidx = gdf.sindex
geoms = gdf.geometry.values
n = len(gdf)
info(f"[3a] Candidate polygons: {n}")
require_shared_edge = True
parent = np.arange(n, dtype=np.int64)

def find(x: int) -> int:
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x

def union(a: int, b: int):
    ra, rb = find(a), find(b)
    if ra != rb:
        parent[rb] = ra

info("[3a] Scanning neighbors via spatial index (predicate='touches')")
scan_t0 = perf_counter()
for i, geom in enumerate(geoms):
    candidates = sidx.query(geom, predicate='touches')
    for j in candidates:
        if j <= i:
            continue
        if not geom.touches(geoms[j]):
            continue
        if require_shared_edge:
            b = geom.boundary.intersection(geoms[j].boundary)
            if b.length <= 0:
                continue
        union(i, j)

scan_t1 = perf_counter()
info(f"[3a] Neighbor scan done ")

roots = np.fromiter((find(i) for i in range(n)), dtype=np.int64, count=n)
patch_id, uniques = pd.factorize(roots)
gdf['patch_id'] = patch_id.astype('int64')
stand_counts = gdf.groupby('patch_id').size().rename('stand_count')

info("[3a] Dissolving touch-connected components to patches")
patches = gdf.dissolve(by='patch_id', as_index=False)
patches = patches.merge(stand_counts, on='patch_id', how='left')

patches2 = patches[patches['stand_count'] >= 2].copy()
info(f"[3a] Patches (>=2 stands): {len(patches2)} of {len(patches)} total")

patches2['area_m2'] = patches2.geometry.area
patches2['area_ha'] = patches2['area_m2'] / 10_000.0
patches2['perim_m'] = patches2.geometry.length

# Shape Index (MSI) for each patch polygon:
# SI = P / (2 * sqrt(pi * A)), minimum is 1 (circle). We compute per patch geometry.
patches2['MSI'] = patches2['perim_m'] / (2.0 * np.sqrt(math.pi * patches2['area_m2']))

patches2.to_parquet(os.path.join(output_dir, "step_3a_patchess_all.parquet"), index=False)  

p75 = patches2["MSI"].quantile(0.75)
info(f"75 quantile {p75}")
patches_less = patches2[(patches2["MSI"] < float(p75)) & (patches2["area_ha"]>=patche_area_3a)]
info(len(patches_less))
patches_less.to_parquet(os.path.join(output_dir, "step_3a.parquet"), index=False)    


In [ ]:
# #NEW step 3b b)	From the layer created in step 3 a), identify which stands occur inside the portions of TSUs and CIA/CIAP
# that fall within a 5km buffer of HV1s. Create layer from selection and call it ‘Old Forest with Functional Characteristics’.

hv1_buff_gdf = gpd.read_parquet(os.path.join(input_parquets, "hv1.parquet"))
tsu = gpd.read_parquet(os.path.join(input_parquets, "tsu_wmb.parquet"))
cia = gpd.read_parquet(os.path.join(input_parquets, "cia_dissolve.parquet"))

try:
    stands = patches_less.copy()
    stands=stands.explode(index_parts=False, ignore_index=False)
except Exception:
    stands = gpd.read_parquet(os.path.join(output_dir, "step_3a.parquet"))
    stands=stands.explode(index_parts=False, ignore_index=False)
    
selecting_features=pd.concat([tsu, cia], ignore_index=True)
selecting_features = selecting_features.explode(index_parts=False).reset_index(drop=True)
info(f"[3b] tsu features: {len(tsu)}")
info(f"[3b] cia features: {len(cia)}")
selecting_features=selecting_features.dissolve()
info(f"[3b] Selecting features: {len(selecting_features)}")

sel = gpd.sjoin(
    stands,
    selecting_features[['geometry']],
    predicate="intersects",
    how="inner"
)

hv1_buff_gdf['buffer_geo']=hv1_buff_gdf.buffer(5000)
hv1_buff_gdf=hv1_buff_gdf.set_geometry('buffer_geo')
info(f"[3b] HV1 buffer bounds: {hv1_buff_gdf.total_bounds.tolist()}")

cols = [c for c in sel.columns if "index_right" in c]
if cols:
    sel = sel.drop(columns=cols)

sel2 = gpd.sjoin(
    sel,
    hv1_buff_gdf[['buffer_geo']],
    predicate="intersects",
    how="inner"
)

out_b = os.path.join(output_dir, "step_3b_Old_Forest_with_Functional_Characteristics.parquet")
sel2.to_parquet(out_b, index=False)

In [ ]:
#step 3c - recruitment forest- c)	From the layer created in step 2 e), select all stands <140. Create a layer from selection entitled ‘Recruitment Forest’ 
sql_3c = f"""
SELECT *,ST_AsText(geometry) as wkt, from '{step2_lyr}'
    WHERE SIFA_2 <140
"""
stands_gdf = return_gdf(sql_3c)
info(f"stands gdf has {len(stands_gdf)} features")


if "FEATURE_ID" in stands_gdf.columns:
    stands_gdf = stands_gdf.rename(columns={"FEATURE_ID": "orig_id"})
else:
    stands_gdf = stands_gdf.reset_index(names="orig_id")
stands_gdf["__row_n__"] = np.arange(len(stands_gdf), dtype=np.int64)


# exploded = stands_gdf.explode(index_parts=False, ignore_index=False)

# exploded["part_id"] = (
#     exploded.groupby("orig_id", dropna=False)["__row_n__"]
#             .rank(method="first")
#             .astype(int) - 1
# )
# exploded.to_parquet(os.path.join(output_dir, "step_3c_Recruitment_Forest.parquet"))


stands_gdf["__row_n__"] = np.arange(len(stands_gdf), dtype=np.int64)

exploded = stands_gdf.explode(index_parts=False, ignore_index=False)

exploded["part_id"] = (
    exploded.groupby("orig_id", dropna=False)["__row_n__"]
            .rank(method="first")
            .astype(int) - 1
)

# minimal, defensive cleanup
if "__index_level_0__" in exploded.columns:
    exploded = exploded.drop(columns="__index_level_0__")

# ensure no duplicated column labels (cheap safeguard)
exploded = exploded.loc[:, ~exploded.columns.duplicated()]

# critical: prevent index from being written as __index_level_0__
exploded.to_parquet(os.path.join(output_dir, "step_3c_Recruitment_Forest.parquet"), index=False)



In [ ]:
#step 3d d)	For the layer created in step 2 e), select all stands not covered by the layers created
# in 3 a) and b). Create a layer from selection entitled ‘Old Forest Without Functional Characteristics’.

sql_3d = f"""
SELECT *,ST_AsText(geometry) as wkt, from '{step2_lyr}'
"""
all_stands_gdf = return_gdf(sql_3d)
info(f"All stands gdf has {len(all_stands_gdf)} features")

if "FEATURE_ID" in all_stands_gdf.columns:
    all_stands_gdf = all_stands_gdf.rename(columns={"FEATURE_ID": "orig_id"})
else:
    all_stands_gdf = all_stands_gdf.reset_index(names="orig_id")
all_stands_gdf["__row_n__"] = np.arange(len(all_stands_gdf), dtype=np.int64)

all_stands_exploded = all_stands_gdf.explode(index_parts=False, ignore_index=False)
all_stands_exploded["part_id"] = (
    all_stands_exploded.groupby("orig_id", dropna=False)["__row_n__"]
            .rank(method="first")
            .astype(int) - 1
)
exploded=gpd.read_file(os.path.join(output_dir, "step_3c_Recruitment_Forest.parquet"))
stands_in_mask=gpd.read_file(os.path.join(output_dir,  "step_3b_Old_Forest_with_Functional_Characteristics.parquet"))
exploded = exploded.loc[:, ~exploded.columns.duplicated()]
mask_layer=pd.concat([exploded, stands_in_mask], ignore_index=True)

ofwofc = gpd.overlay(all_stands_exploded, mask_layer, how="difference", keep_geom_type=False)
ofwofc.to_parquet(os.path.join(output_dir, "step_3d_Old_Forest_without_Functional_Characteristics.parquet"), index=False)



In [ ]:
#step 3E Union non-THLB with BRFN/BC cost surface Layers. Delete portions of cost surface that overlap non-THLB. Create layer from resultant entitled ‘Low-Cost Lands’
non_thlb_query=f"""SELECT *,ST_AsText(geometry) as wkt, from '{aflb_parquet}' where thlb_fact is Null OR thlb_fact = 0
"""
non_thlb = return_gdf(non_thlb_query)
info(f"Non-THLB gdf has {len(non_thlb)} features")

non_thlb.to_parquet(os.path.join(output_parquets,'non_thlb.parquet'))

cost_surface_path=os.path.join(source_data, "blue_berry_wmb_geoparq","cost_surface.parquet")
cost_surface=gpd.read_parquet(cost_surface_path)
info('parquet read') 

hits = gpd.sjoin(
    non_thlb[["geometry"]],
    cost_surface[["geometry"]],
    how="inner",
    predicate="intersects"
)

drop_idx = hits.index.unique()
low_cost = non_thlb.loc[~non_thlb.index.isin(drop_idx)].copy()

out_fp = os.path.join(output_dir, "step_3e_Low_Cost_Lands.parquet")
low_cost.to_parquet(out_fp, index=False)

info(f"Step 3E: Removed {len(drop_idx)} non_THLB features intersecting cost surface; "
     f"kept {len(low_cost)} → {out_fp}")

In [ ]:
#Step 3F f)	Create WVM   with no forest values. Assign highest multipliers to CIA1/CIAP, Cabins, and Mineral Licks. Assign next high multipliers
# assigned to CIA2 and Low-Cost Lands. Create a layer from the top 25% WVM and title it ‘Optimal Lands’
if "cia" not in globals():
    cia=gpd.read_parquet(os.path.join(wmv_inputs,'cia_1_p.parquet'))
    info('cia 1 p')
if "licks" not in globals():
    licks=gpd.read_parquet(os.path.join(wmv_inputs,'mineral_licks.parquet'))
    info('licks')
if "cia2" not in globals():
    cia2 =gpd.read_parquet(os.path.join(wmv_inputs,'cia_2.parquet'))
    info('cia 2')
if "low_cost" not in globals():
    low_cost = gpd.read_parquet(os.path.join(output_dir, "step_3e_Low_Cost_Lands.parquet"))
    low_cost=low_cost.dissolve()
    info('low cost')
if "headwater_gdf" not in globals():
    headwater_gdf=gpd.read_parquet(os.path.join(wmv_inputs,'headwaters.parquet'))
    info('headwaters')
if "riparian_gdf" not in globals():
    riparian_gdf=gpd.read_parquet(os.path.join(wmv_inputs,'riparian.parquet'))
    info('rip')
if "wetlands_gdf" not in globals():
    wetlands_gdf=gpd.read_parquet(os.path.join(wmv_inputs,'wetlands.parquet'))
    info('wetlands')
if "moose_gdf" not in globals():
    moose_gdf=gpd.read_parquet(os.path.join(wmv_inputs,'moose.parquet'))
    info('moose')
if "caribou1" not in globals():
    caribou1=gpd.read_parquet(os.path.join(wmv_inputs,'caribou_1.parquet'))
    info('caribou 1')
if "caribou2" not in globals():
    caribou2=gpd.read_parquet(os.path.join(wmv_inputs,'caribou_2.parquet'))
    info('caribou 2')
if "fisher" not in globals():
    fisher=gpd.read_parquet(os.path.join(wmv_inputs,'fisher.parquet'))

grid = gpd.read_parquet(grid_path)

layers = {
    "CIA1_CIAP": (cia, 1.00),
    "Licks":     (licks, 1.00),
    "CIA2":      (cia2, 0.75),
    "LowCost":   (low_cost, 0.75),
    "Wetlands":  (wetlands_gdf, 0.75),
    "Moose":     (moose_gdf, 0.75),
    "Riparian":  (riparian_gdf, 0.50),
    "Caribou1":  (caribou1, 0.50),
    "Headwaters":(headwater_gdf, 0.25),
    "Caribou2":  (caribou2, 0.25),
    "Fisher":    (fisher, 0.25),
}

grid_out, optimal_cells, thr = build_wvm_additive_cells_tiled_parallel(
    grid,
    layers,
    tile_km=10.0,            
    weight_mode="proportion", 
    simplify_tolerance=None,  
    precision_snap=1.0,
    non_forest_mask=None,
    max_workers=None,      
    save_intermediate_tile_weights=False,
    output_dir=output_dir
)

opt_path = os.path.join(output_dir, "step_3f_optimal_lands_cells.parquet")

wmb = gpd.read_parquet(os.path.join(input_parquets, "priority_WMB.parquet"))
info('wmb polygons read')
optimal_lands=gpd.clip(optimal_cells,wmb)
thr = optimal_lands["weighted_ha"].quantile(0.75)
optimal_cells = optimal_lands.loc[optimal_lands["weighted_ha"] >= thr].copy()
opt_path = os.path.join(output_dir, "step_3f_optimal_lands_cells.parquet")
optimal_cells.to_parquet(opt_path, index=False)


In [ ]:
wmb=gpd.read_parquet(os.path.join(input_parquets,'priority_wmb.parquet'))

In [ ]:
#step3g Overlay WMB with 3ha hexagon grid with identifiers for each cell. Seek to connect all ‘Recruitment Forest’
# to ‘Old Forest with Functional Characteristics’ by selecting adjoining grid cells containing:
#   -Old forest without functional characteristics 
#   -Optimal lands 
# if 'grid' not in globals():
grid=gpd.read_parquet(grid_path)
info('load grid')
# if 'wmb' not in globals():
wmb= gpd.read_parquet(os.path.join(input_parquets, "priority_WMB.parquet"))
info('wmb polygons read')
wmb_grid=gpd.sjoin(grid,wmb, how='inner')
# if not os.path.exists(os.path.join(output_dir, 'step_3g_grid.parquet')):
wmb_grid_path=os.path.join(output_dir, 'step_3g_grid.parquet')
wmb_grid.to_parquet(wmb_grid_path)
info('wmb grid created')
# if 'rec_for' not in globals():
rec_for=gpd.read_parquet(rec_for_path)
info('rec for read')
# if 'func_for' not in globals():
func_for=gpd.read_parquet(os.path.join(output_dir, "step_3b_Old_Forest_with_Functional_Characteristics.parquet"))
info('func for read')
# if 'no_func_for' not in globals():
no_func_for=gpd.read_parquet(os.path.join(output_dir, "step_3d_Old_Forest_without_Functional_Characteristics.parquet"))
info('no func for read')
# if 'optimal_lands' not in globals():
optimal_lands=gpd.read_parquet(os.path.join(output_dir, "step_3f_optimal_lands_cells.parquet"))
info('opt land read ')

grid = to3005(grid)
wmb = to3005(wmb)
rec_for = to3005(rec_for)
func_for = to3005(func_for)
no_func_for = to3005(no_func_for)
optimal_lands = to3005(optimal_lands)

In [ ]:
#first go at connecting adjacent cells 

# rec_ids  = intersect_ids(wmb_grid, rec_for, grid_id_col)
# func_ids = intersect_ids(wmb_grid, func_for, grid_id_col)

# old_all_ids = intersect_ids(wmb_grid, con_for, grid_id_col)
# old_fc_ids  = func_ids  # by definition
# old_no_fc_ids = old_all_ids - old_fc_ids

# opt_ids = intersect_ids(wmb_grid, optimal_lands, grid_id_col) if not optimal_lands.empty else set()
# allowed_ids = old_no_fc_ids.union(opt_ids)

# node_ids = set().union(rec_ids, func_ids, allowed_ids)
# nodes_gdf = wmb_grid[wmb_grid[grid_id_col].isin(node_ids)].copy()

# # Build spatial index for the candidate nodes and start network
# nodes_gdf = nodes_gdf.reset_index(drop=True)
# nodes_gdf["__idx"] = np.arange(len(nodes_gdf))

# sidx = nodes_gdf.sindex
# geoms = nodes_gdf.geometry.values
# ids   = nodes_gdf[grid_id_col].values

# # Build edges where cells TOUCH (hex adjacency)
# edges = []
# for i, geom in enumerate(geoms):
#     # bbox candidates
#     cand_idx = sidx.query(geom, predicate="intersects")
#     for j in cand_idx:
#         if j <= i:
#             continue
#         if geom.touches(geoms[j]):  # “adjoining” cells
#             edges.append((ids[i], ids[j]))

# G = nx.Graph()
# G.add_nodes_from(ids)
# G.add_edges_from(edges)

# #Multi‑source BFS
# func_nodes = set(id for id in ids if id in func_ids)
# rec_nodes  = set(id for id in ids if id in rec_ids)
# node_ids = rec_nodes | func_nodes | allowed_ids
# Restrict traversal: through allowed cells OR rec/func themselves
# def traversable(n):
#     return (n in allowed_ids) or (n in rec_nodes) or (n in func_nodes)

# # Multi-source BFS from func nodes
# visited = {}
# parent = {}
# q = deque()

# for f in func_nodes:
#     visited[f] = True
#     parent[f] = None
#     q.append(f)

# while q:
#     u = q.popleft()
#     for v in G.neighbors(u):
#         if v in visited:
#             continue
#         if not traversable(v):
#             continue
#         visited[v] = True
#         parent[v] = u
#         q.append(v)

# Reconstruct minimal paths for each rec node that was reached
# path_nodes = set()
# unreached  = []

# for r in rec_nodes:
#     if r not in visited:
#         unreached.append(r)
#         continue
#     # backtrack to a func node
#     cur = r
#     while cur is not None:
#         path_nodes.add(cur)
#         cur = parent[cur]


# Subgraph induced by all relevant nodes (rec + func + allowed)
# H = G.subgraph(node_ids).copy()

# # Connected components
# components = list(nx.connected_components(H))

# selected_ids_expanded = set()
# unreached = []

# for comp in components:
#     has_rec  = len(comp.intersection(rec_nodes))  > 0
#     has_func = len(comp.intersection(func_nodes)) > 0
#     if has_rec and has_func:
#         # If a component contains both endpoints, keep the ENTIRE contiguous mass
#         selected_ids_expanded |= comp

# # Any recruitment cells not in a connecting component are unreached
# covered_rec = selected_ids_expanded.intersection(rec_nodes)
# unreached = list(rec_nodes - covered_rec)

# # Final selection: all cells in connecting components
# connecting_cells = wmb_grid[wmb_grid[grid_id_col].isin(selected_ids_expanded)].copy()
# connecting_cells_path = os.path.join(output_dir, "step_3g_connecting_cells.parquet")
# connecting_cells.to_parquet(connecting_cells_path, index=False)

# info(f"[3g] rec={len(rec_nodes)}, func={len(func_nodes)}, allowed={len(allowed_ids)}")
# info(f"[3g] selected cells (component-expanded): {len(connecting_cells)}; unreached rec: {len(unreached)}")

# # QA: Save unreached recruitment cells
# if unreached:
#     unreached_gdf = wmb_grid[wmb_grid[grid_id_col].isin(unreached)].copy()
#     unreached_gdf.to_parquet(os.path.join(output_dir, "step_3g_unreached_recruitment_cells.parquet"), index=False)


In [ ]:

# Thresholds
MIN_PROP_ALLOWED  = 0.05
MIN_PROP_REC_FUNC = 0.01 


# LOSE: maximize connectivity (intersects)
rec_loose  = ids_by_intersects(wmb_grid, rec_for,  grid_id_col)
func_loose = ids_by_intersects(wmb_grid, func_for, grid_id_col)
old_loose  = ids_by_intersects(wmb_grid, no_func_for,   grid_id_col)
opt_loose = ids_by_intersects(wmb_grid, optimal_lands, grid_id_col)

# allowed_loose = old_loose | opt_loose
#20260414 above olde below new
allowed_loose = old_loose & opt_loose

# STRICT: clean clusters (coverage)
rec_strict,  rec_props   = ids_by_coverage(wmb_grid, rec_for,       grid_id_col, MIN_PROP_REC_FUNC)
func_strict, func_props  = ids_by_coverage(wmb_grid, func_for,      grid_id_col, MIN_PROP_REC_FUNC)
old_strict,  old_props   = ids_by_coverage(wmb_grid, no_func_for,   grid_id_col, MIN_PROP_ALLOWED)
opt_strict,  opt_props   = ids_by_coverage(wmb_grid, optimal_lands, grid_id_col, MIN_PROP_ALLOWED)
allowed_strict = old_strict | opt_strict


# Candidate node sets
node_ids_loose  = rec_loose | func_loose | allowed_loose
node_ids_strict = rec_strict | func_strict | allowed_strict

nodes_loose_gdf = (wmb_grid[wmb_grid[grid_id_col].isin(node_ids_loose)]
                   .drop_duplicates(subset=[grid_id_col])
                   .reset_index(drop=True))

# Adjacency via STRtree.query_bulk
try:
    pairs = nodes_loose_gdf.sindex.query_bulk(nodes_loose_gdf.geometry, predicate="touches")
    li, ri = pairs
    mask = li < ri
    li, ri = li[mask], ri[mask]
    edges = list(zip(
        nodes_loose_gdf.iloc[li][grid_id_col].to_numpy(),
        nodes_loose_gdf.iloc[ri][grid_id_col].to_numpy()
    ))
except Exception:
    pairs_df = gpd.sjoin(
        nodes_loose_gdf[[grid_id_col, "geometry"]].rename(columns={grid_id_col: "id_l"}),
        nodes_loose_gdf[[grid_id_col, "geometry"]].rename(columns={grid_id_col: "id_r"}),
        predicate="touches",
        how="inner",
    )[["id_l", "id_r"]]
    pairs_df = pairs_df.loc[pairs_df["id_l"] < pairs_df["id_r"]]
    edges = list(pairs_df.itertuples(index=False, name=None))

G_loose = nx.Graph()
G_loose.add_nodes_from(nodes_loose_gdf[grid_id_col].to_numpy())
G_loose.add_edges_from(edges)


# Seeds for the strict network (can be large): all strict nodes (allowed + endpoints)
strict_seed = set(node_ids_strict) & set(node_ids_loose)  
endpoints   = set(rec_loose) | set(func_loose)       

# Multi-source BFS FROM strict_seed THROUGH loose graph; add shortest paths to endpoints
visited = {}
parent = {}
q = deque()

for s in strict_seed:
    visited[s] = True
    parent[s] = None
    q.append(s)

found_endpoints = set()
path_nodes = set()

while q:
    u = q.popleft()
    if (u in endpoints) and (u not in strict_seed) and (u not in found_endpoints):
        found_endpoints.add(u)
        cur = u
        while cur is not None:
            path_nodes.add(cur)
            cur = parent[cur]

    for v in G_loose.neighbors(u):
        if v in visited:
            continue
        if not ((v in allowed_loose) or (v in endpoints) or (v in strict_seed)):
            continue
        visited[v] = True
        parent[v] = u
        q.append(v)

# Final HYBRID set: strict network + minimal connectors + all endpoints
hybrid_ids = set(strict_seed) | path_nodes | endpoints

H_loose = G_loose.subgraph(node_ids_loose).copy()
loose_components = []
loose_selected = set()
for comp in nx.connected_components(H_loose):
    if (comp & set(rec_loose)) and (comp & set(func_loose)):
        loose_selected |= comp



connecting_cells_hybrid = wmb_grid[wmb_grid[grid_id_col].isin(hybrid_ids)].copy()
# connecting_cells_hybrid.to_parquet(os.path.join(output_dir, "step_3g_connecting_cells_hybrid.parquet"), index=False)
strict_only_ids = node_ids_strict | (set(rec_loose) & node_ids_loose) | (set(func_loose) & node_ids_loose)
connecting_cells_strict = wmb_grid[wmb_grid[grid_id_col].isin(strict_only_ids)].copy()
# connecting_cells_strict.to_parquet(os.path.join(output_dir, "step_3g_connecting_cells_strict_seed.parquet"), index=False)
connecting_cells_loose = wmb_grid[wmb_grid[grid_id_col].isin(loose_selected)].copy()
# connecting_cells_loose.to_parquet(os.path.join(output_dir, "step_3g_connecting_cells_loose_components.parquet"), index=False)
# Connector-only cells = hybrid minus strict and endpoints
connector_ids = hybrid_ids - strict_only_ids
connector_gdf = wmb_grid[wmb_grid[grid_id_col].isin(connector_ids)].copy()
# connector_gdf.to_parquet(os.path.join(output_dir, "step_3g_connector_cells_minimal.parquet"), index=False)

if connector_gdf.empty:
    corridor = gpd.GeoDataFrame(columns=["geometry"], geometry="geometry", crs=wmb_grid.crs)
else:
    corridor = connector_gdf.assign(grp=1).dissolve(by="grp", as_index=False)[["geometry"]]
# corridor.to_parquet(os.path.join(output_dir, "step_3g_connecting_corridor_hybrid.parquet"), index=False)

print(
    f"[3g HYBRID] loose nodes={len(node_ids_loose)} | strict nodes={len(node_ids_strict)} | "
    f"hybrid cells={len(connecting_cells_hybrid)} | strict_seed={len(strict_seed)} | "
    f"endpoints={len(endpoints)} | minimal connectors={len(connector_gdf)}"
)

# The hybrid approach
Think of the hex grid like tiles on a floor.


1. Find where the tiles touch (adjacent hexes). That lets us see which tiles can connect to which other tiles.


2. Identify the key tiles:

    - Endpoints: tiles with Recruitment forest and Functional old forest (our “from” and “to”).
    - Connecting fabric: tiles that contain enough Old‑without‑Functional forest or Optimal Lands.



3. Build two versions of the connecting fabric:

    - Loose (generous): a tile counts if it intersects any bit of the fabric. This understands where connections are possible.
    - Strict (clean): a tile counts only if a meaningful share (e.g., ≥10%) of the tile is covered by the fabric. This keeps the visible/real habitat and filters out slivers.



4. Start with the strict network: Take all strict fabric tiles (plus Recruitment and Functional tiles). This is our clean base.


5. Add only what’s needed to connect: If any Recruitment or Functional tile is not connected to the strict network because strict rules created tiny gaps, we bridge the gap with the shortest possible chain of loose tiles.

    - In other words: add the minimum number of tiles needed to connect endpoints, but only through places we know can connect (the loose fabric).



6. Result = strict + minimal connectors + endpoints
You get the full, meaningful fabric wherever coverage is good, and small, necessary connectors where the strict filter would have accidentally broken the linkage.

# Why this is defensible

- Connectivity is guaranteed: If a connection exists, we keep it (via minimal connectors).
- Habitat is meaningful: Most connecting tiles meet a coverage threshold (e.g., ≥10%); the map won’t be peppered with “empty” tiles.
- Corridors aren’t bloated: We don’t pull in huge areas just because they’re technically connected. We add only what we need.

This hybrid logic reflects practical planning: use the strongest habitat whenever possible, and only “step across” thinner habitat when it’s the shortest/necessary route to join key areas.


# How to explain it in one sentence

```We connect Recruitment to Functional habitat across adjoining hexes by keeping all meaningful habitat tiles (where coverage is solid), and—only where needed—adding the shortest chain of tiles that can connect them, so corridors stay continuous and clean.```

In [ ]:
hybrid_ids = set(connecting_cells_hybrid[grid_id_col].unique())
idx = pd.Index(sorted(hybrid_ids), name=grid_id_col)

def reindex_props(props_df):
    if props_df is None or props_df.empty:
        return pd.Series(0.0, index=idx, name="prop")
    return props_df.reindex(idx)["prop"].fillna(0.0)

prop_rec  = reindex_props(rec_props)
prop_func = reindex_props(func_props)
prop_old  = reindex_props(old_props)
prop_opt  = reindex_props(opt_props)

cell_attr = pd.DataFrame({
    "prop_rec":  prop_rec.values,
    "prop_func": prop_func.values,
    "prop_old":  prop_old.values,
    "prop_opt":  prop_opt.values,
}, index=idx)

# 2) Threshold flags (consistent with your hybrid thresholds) 
cell_attr["is_rec"]     = cell_attr["prop_rec"]  >= MIN_PROP_REC_FUNC
cell_attr["is_func"]    = cell_attr["prop_func"] >= MIN_PROP_REC_FUNC
cell_attr["is_old"]     = cell_attr["prop_old"]  >= MIN_PROP_ALLOWED
cell_attr["is_opt"]     = cell_attr["prop_opt"]  >= MIN_PROP_ALLOWED
cell_attr["is_allowed"] = cell_attr["is_old"] | cell_attr["is_opt"]

# 3) Primary source & summary of sources above threshold 
# Priority for ties: REC > FUNC > OLD > OPT
def primary_source(row):
    vals = {
        "REC":  row["prop_rec"],
        "FUNC": row["prop_func"],
        "OLD":  row["prop_old"],
        "OPT":  row["prop_opt"],
    }
    priority = {"REC": 0, "FUNC": 1, "OLD": 2, "OPT": 3}
    best = sorted(vals.items(), key=lambda kv: (-kv[1], priority[kv[0]]))[0]
    return best[0], best[1]
ps = cell_attr.apply(primary_source, axis=1, result_type="expand")
cell_attr["primary_source"] = ps[0]
cell_attr["prop_primary"]   = ps[1].astype(float)

def source_summary(row):
    parts = []
    if row["is_rec"]:  parts.append("REC")
    if row["is_func"]: parts.append("FUNC")
    if row["is_old"]:  parts.append("OLD")
    if row["is_opt"]:  parts.append("OPT")
    return ";".join(parts) if parts else ""

cell_attr["sources_present"] = cell_attr.apply(source_summary, axis=1)

#  4) Hybrid connection role & flags
# Build typed sets for fast membership checks
cast = connecting_cells_hybrid[grid_id_col].dtype.type
allowed_strict_set = {cast(i) for i in allowed_strict} if isinstance(allowed_strict, set) else set(allowed_strict)
connector_ids_set  = {cast(i) for i in connector_ids}  if isinstance(connector_ids, set)  else set(connector_ids)
endpoints_set      = {cast(i) for i in (set(rec_loose) | set(func_loose))}

# Flags
cell_attr["in_strict_allowed"] = cell_attr.index.isin(allowed_strict_set)
cell_attr["in_connector"]      = cell_attr.index.isin(connector_ids_set)
cell_attr["in_endpoints"]      = cell_attr.index.isin(endpoints_set)

# Role rule:
#  - If both REC & FUNC present: REC_AND_FUNC
#  - Else if REC: REC
#  - Else if FUNC: FUNC
#  - Else if strict allowed: ALLOWED_STRICT
#  - Else if connector: CONNECTOR_MIN
#  - Else: "" (shouldn't occur in hybrid, but defensive)
def role_label(row):
    if row["is_rec"] and row["is_func"]:
        return "REC_AND_FUNC"
    if row["is_rec"]:
        return "REC"
    if row["is_func"]:
        return "FUNC"
    if row["in_strict_allowed"]:
        return "ALLOWED_STRICT"
    if row["in_connector"]:
        return "CONNECTOR_MIN"
    return ""
cell_attr["connection_role"] = cell_attr.apply(role_label, axis=1)
cell_attr["prop_allowed_total"] = (cell_attr["prop_old"] + cell_attr["prop_opt"]).clip(upper=1.0)
cell_attr["prop_endpoint_total"] = (cell_attr["prop_rec"] + cell_attr["prop_func"]).clip(upper=1.0)

# 5) Attach attributes to the hybrid GeoDataFrame & save
hybrid_attr = connecting_cells_hybrid.merge(
    cell_attr.reset_index(), on=grid_id_col, how="left"
)

hybrid_attr_path = os.path.join(output_dir, "step_3g_connecting_cells_hybrid_attributed.parquet")
hybrid_attr.to_parquet(hybrid_attr_path, index=False)


In [ ]:
# Step 3h Starting from a cell containing either Recruitment Forest, or Old Forest with Functional Characteristics,
# begin to select 4 cells in all outward directions until you encounter a cell where none of the criteria in 3)g are present,
# there is not a patch of Recruitment Forest or Old Forest with Functional Characteristics,
# or you run into Critical/Electrification Infrastructure. As a last step, connect selected cells outside HV1 boundaries
# to any HV1 boundary within 4 cells (except for if there is critical infrastructure in-between). 
if 'hv1' not in globals():
    info('loading hv1')
    hv1 = gpd.read_parquet(os.path.join(input_parquets, "hv1.parquet"))
 
if 'grid_3g' not in globals():
    info('loading grid')
    grid_3g=gpd.read_parquet(os.path.join(output_dir, 'step_3g_connecting_cells_hybrid_attributed.parquet'))

if 'rec_for' not in globals():
    info('loading rec for')
    rec_for=gpd.read_parquet(rec_for_path)
    
if 'func_for' not in globals():
    info('loading func for')
    func_for=gpd.read_parquet(os.path.join(output_dir, "step_3b_Old_Forest_with_Functional_Characteristics.parquet"))
if 'grid_full' not in globals():
    info('loading grid full')
grid_full=gpd.read_parquet(os.path.join(input_parquets,'hexagons_3ha.parquet'))
grid_id_col='GRID_ID'
#set barriers gdf 
barriers_gdf=gpd.read_parquet(os.path.join(input_parquets,'barriers.parquet'))

# # Outward selection using 3g mask over full grid; HV1 Development acts as a barrier ---
# max_rings = 4

if "barriers_gdf" in globals() and isinstance(barriers_gdf, gpd.GeoDataFrame):
    barriers_gdf = to3005(fix(barriers_gdf))
else:
    barriers_gdf = gpd.GeoDataFrame(columns=["geometry"], geometry="geometry", crs=grid_full.crs)

# Deduplicate by hex id
grid_full = grid_full.drop_duplicates(subset=[grid_id_col]).reset_index(drop=True)
grid_3g   = grid_3g.drop_duplicates(subset=[grid_id_col]).reset_index(drop=True)

allowed_3g_ids = set(grid_3g[grid_id_col].to_numpy())
info(f"Full grid cells: {len(grid_full)} | 3g allowed cells: {len(allowed_3g_ids)}")
# 1) Seeds and base barriers

#new change 20260414 adding in new seeds, old forest that overlaps Optimal Lands (any overlapp allowed)
old_no_func_opt_ids = (
    ids_by_intersects(grid_full, no_func_for,   grid_id_col)
    & ids_by_intersects(grid_full, optimal_lands, grid_id_col)
)


# seed_hv1_ids  = ids_by_intersects(grid_full, hv1, grid_id_col)
seed_rec_ids  = ids_by_intersects(grid_full, rec_for, grid_id_col)
seed_func_ids = ids_by_intersects(grid_full, func_for, grid_id_col)

# seed_ids = set().union( seed_rec_ids, seed_func_ids) #seed_hv1_ids,
# info(f"Seed cells (HV1 ∪ Rec ∪ Func): {len(seed_ids)}")

# above old seed_ids line replaced by new line below to add in old_no_func_opt_ids as seeds (any old forest that overlaps optimal lands) 20260414
seed_ids = (
    set(seed_rec_ids)
    | set(seed_func_ids)
    | set(old_no_func_opt_ids)
)

info(
    f"Seed cells by class — "
    f"REC: {len(seed_rec_ids)}, "
    f"FUNC: {len(seed_func_ids)}, "
    f"OLD–NOFUNC∩OPT: {len(old_no_func_opt_ids)}, "
    f"TOTAL: {len(seed_ids)}"
)


barrier_ids = ids_by_intersects(grid_full, barriers_gdf, grid_id_col) if not barriers_gdf.empty else set()

# 2) Add HV1 'Development' zone as additional barriers
zone_col = next((c for c in hv1.columns if c.lower() == "zone"), None)
if zone_col:
    hv1_dev = hv1[hv1[zone_col].astype(str).str.strip().str.upper() == "DEVELOPMENT"].copy()
    hv1_dev_ids = ids_by_intersects(grid_full, hv1_dev, grid_id_col) if not hv1_dev.empty else set()
    info(f"HV1 Development cells (barriers): {len(hv1_dev_ids)}")
    barrier_ids |= hv1_dev_ids
else:
    info("HV1 'Zone' column not found; no Development barriers added.")

info(f"Barrier cells (infra + HV1 Development): {len(barrier_ids)}")

# Remove seeds that fall in barriers (cannot start in a barrier cell)
seed_ids = [i for i in seed_ids if i not in barrier_ids]
info(f"Seeds eligible to start expansion: {len(seed_ids)}")

# 3) Build adjacency graph on entire grid (touching neighbours)
nodes_all = grid_full.reset_index(drop=True)

edges = []
try:
    geoms = nodes_all.geometry.to_numpy()
    tree = STRtree(geoms)
    idx_src, idx_tgt = tree.query(geoms, predicate="touches")
    m = idx_src < idx_tgt
    idx_src, idx_tgt = idx_src[m], idx_tgt[m]
    left_ids  = nodes_all.iloc[idx_src][grid_id_col].to_numpy()
    right_ids = nodes_all.iloc[idx_tgt][grid_id_col].to_numpy()
    edges = list(zip(left_ids, right_ids))
    info(f"Adjacency edges (STRtree touches): {len(edges)}")
except Exception as e:
    info(f"STRtree touches failed ({e}); falling back to sjoin (slower).")
    pairs_df = gpd.sjoin(
        nodes_all[[grid_id_col, "geometry"]].rename(columns={grid_id_col: "id_l"}),
        nodes_all[[grid_id_col, "geometry"]].rename(columns={grid_id_col: "id_r"}),
        predicate="touches",
        how="inner",
    )[["id_l", "id_r"]]
    pairs_df = pairs_df.loc[pairs_df["id_l"] < pairs_df["id_r"]]
    edges = list(pairs_df.itertuples(index=False, name=None))
    info(f"Adjacency edges (sjoin): {len(edges)}")

G = nx.Graph()
G.add_nodes_from(nodes_all[grid_id_col].to_numpy())
G.add_edges_from(edges)

# # 4) BFS outward (≤ max_rings) constrained by 3g + barriers (incl. HV1 Development)
# visited = {}  
# q = deque()
# for s in seed_ids:
#     visited[s] = 0
#     q.append(s)

# def traversable(cell_id: int) -> bool:
#     if cell_id in barrier_ids:
#         return False
#     if cell_id not in allowed_3g_ids:
#         return False
#     return True

# while q:
#     u = q.popleft()
#     ring_u = visited[u]
#     if ring_u >= max_rings:
#         continue
#     for v in G.neighbors(u):
#         if v in visited:
#             continue
#         if not traversable(v):
#             continue
#         visited[v] = ring_u + 1
#         q.append(v)

# 5) HV1-band bridging (<=4 rings from HV1; respect barriers; ignore 3g)
#     Build HV1 band; find minimal corridors between Step-4 components within band;
#     backfill corridors to ring-1 only when HV1 is involved; augment sel_ids.
info(f"[3h-bridge] Deriving HV1 outward band (<={max_rings} rings)")

# All cells that intersect any HV1 polygon (all zones)
hv1_all_ids = ids_by_intersects(grid_full, hv1, grid_id_col)

# Partition Dev vs Non-Dev (reuse hv1_dev_ids if computed earlier)
hv1_dev_ids = locals().get("hv1_dev_ids", set())
if (not hv1_dev_ids) and zone_col:
    hv1_dev = hv1[hv1[zone_col].astype(str).str.strip().str.upper() == "DEVELOPMENT"].copy()
    hv1_dev_ids = ids_by_intersects(grid_full, hv1_dev, grid_id_col) if not hv1_dev.empty else set()
hv1_nondev_ids = hv1_all_ids - hv1_dev_ids

# BFS outward from HV1 Non-Development outside edge up to 4 rings; do not enter HV1 nor barriers
ring_dist = {}          
parent = {}         
q = deque()

# Ring 1: neighbors of HV1 Non-Development that are not HV1 and not barriers
ring1 = set()
for h in hv1_nondev_ids:
    for nb in G.neighbors(h):
        if nb in hv1_all_ids:
            continue
        if nb in barrier_ids:
            continue
        ring1.add(nb)

# Seed ring-1
for h in ring1:
    if h not in ring_dist:
        ring_dist[h] = 1
        parent[h] = None
        q.append(h)

# Rings 2-4
while q:
    u = q.popleft()
    d = ring_dist[u]
    if d >= 4:
        continue
    for v in G.neighbors(u):
        if v in ring_dist:
            continue
        if v in hv1_all_ids:
            continue
        if v in barrier_ids:
            continue
        ring_dist[v] = d + 1
        parent[v] = u
        q.append(v)

hv1_band_ids = {h for h, d in ring_dist.items() if 1 <= d <= 4}
info(f"[3h-bridge] HV1 band cells (<=4 rings, excl. barriers/HV1): {len(hv1_band_ids)}")

# 6) established selection and components
sel_ids = set(visited.keys()) | set(seed_ids)
H_sel = G.subgraph(sel_ids)
components = list(nx.connected_components(H_sel))
num_comps = len(components)
comp_labels = {n: idx for idx, comp in enumerate(components) for n in comp}
info(f"[3h-bridge] Step-4 components: {num_comps}")



# 7)-bridge Combined bridging within HV1 band (<= max_rings; barriers respected)
#   A) HV1-anchored: from HV1 (incl. that HV1 cell) outward (<= max_rings) to join Step-4 rings
#   B) Ring-to-Ring: between different Step-4 components (<= max_rings) using minimal corridors
from collections import defaultdict

sel_ids0 = set(sel_ids) 
MAX_BRIDGE_LEN = max_bridging
optimal_ids = ids_by_intersects(grid_full, optimal_lands, grid_id_col)
#  A) HV1-anchored bridging
# We expand from ring-1 band cells (adjacent to HV1 Non-Dev) up to <= max_rings,
# and when we touch the Step-4 selection, we add the shortest corridor and the HV1 endpoint cell(s).

# Precompute: for each ring-1 band cell, which HV1 Non-Dev cells it touches
ring1_to_hv1 = defaultdict(set)
for b in ring1:
    for nb in G.neighbors(b):
        if nb in hv1_nondev_ids:
            ring1_to_hv1[b].add(nb)
# Multi-source BFS from all ring-1 seeds with per-seed parents
parents_hv1 = {s: {s: None} for s in ring1}
dist_hv1 = {s: {s: 0} for s in ring1}
q_hv1 = deque((s, s) for s in ring1)  # (node, seed)

bridge_paths_hv1 = []   # list of sets (band nodes in each corridor)
hv1_endpoints_to_add = set()
while q_hv1:
    u, seed = q_hv1.popleft()
    du = dist_hv1[seed][u]
    if du >= MAX_BRIDGE_LEN:
        continue
    # If 'u' is adjacent to any Step-4 selected cell, found a corridor
    touches_sel = False
    meet_node = None
    for nb in G.neighbors(u):
        if nb in sel_ids0:
            touches_sel = True
            meet_node = u
            break

    if touches_sel:
        # Reconstruct shortest path (band) from seed (ring-1) to meet_node
        path = []
        cur = meet_node
        while cur is not None:
            path.append(cur)
            cur = parents_hv1[seed][cur]
        path.reverse()
        corridor = set(path)
        if len(corridor) <= MAX_BRIDGE_LEN:
            bridge_paths_hv1.append(corridor)
            # Include the HV1 endpoint cell(s) touching the ring-1 seed
            hv1_endpoints_to_add |= ring1_to_hv1.get(seed, set())
        # Do not expand further from this
        continue
    # Otherwise, expand within the band
    for v in G.neighbors(u):
        if v not in hv1_band_ids:
            continue
        if v in parents_hv1[seed]:
            continue
        parents_hv1[seed][v] = u
        dist_hv1[seed][v] = du + 1
        q_hv1.append((v, seed))

bridges_hv1 = set().union(*bridge_paths_hv1) if bridge_paths_hv1 else set()
info(f"[3h-bridge A] HV1->Ring corridors: {len(bridge_paths_hv1)} | band nodes={len(bridges_hv1)} | hv1_endpoints={len(hv1_endpoints_to_add)}")

# B) Ring-to-Ring bridging
# Seeds are band nodes that touch the current Step-4 selection; label each by the component id
border_by_comp: dict[int, set] = defaultdict(set)
for b in hv1_band_ids:
    if b in barrier_ids:
        continue

    labs = set()
    for nb in G.neighbors(b):
        if nb in sel_ids0:
            labs.add(comp_labels[nb])
    for lab in labs:
        border_by_comp[lab].add(b)

active_labels = [lab for lab, nodes in border_by_comp.items() if nodes]
info(f"[3h-bridge B] Components touching HV1 band: {len(active_labels)}")

# Minimal corridors by frontier-meet (multi-source, multi-label BFS over band)
visited_by = defaultdict(set)
prev_by_label = {lab: {} for lab in active_labels}
q_rr = deque()

for lab in active_labels:
    for s in border_by_comp[lab]:
        if lab not in visited_by[s]:
            visited_by[s].add(lab)
            prev_by_label[lab][s] = None
            q_rr.append((s, lab, 0))  # node, label, dist

def _reconstruct(prev_map: dict, m: int) -> list[int]:
    path = []
    cur = m
    while cur is not None:
        path.append(cur)
        cur = prev_map.get(cur, None)
    path.reverse()
    return path

bridge_paths_rr = []

while q_rr:
    u, lab, d = q_rr.popleft()
    if d >= MAX_BRIDGE_LEN:
        continue

    # Frontier meet with a different label -> build corridor
    others = visited_by[u] - {lab}
    if others:
        other = next(iter(others))
        path_a = _reconstruct(prev_by_label[lab], u)
        path_b = _reconstruct(prev_by_label[other], u)
        # corridor = set(path_a) | set(path_b)
        # if len(corridor) <= MAX_BRIDGE_LEN:
        #     bridge_paths_rr.append(corridor)
        
        #20260414 change: enforce max length on the combined path, not just the union of nodes (since they can overlap at the meet node)
        combined_len = len(path_a) + len(path_b) - 1  # subtract shared meet node

        if combined_len <= MAX_BRIDGE_LEN:
            corridor = set(path_a) | set(path_b)
            bridge_paths_rr.append(corridor)

        continue

    # Expand within band
    for v in G.neighbors(u):
        if v not in hv1_band_ids:
            continue
        if lab in visited_by[v]:
            continue
        if v in barrier_ids:
                continue    
        if lab in visited_by[v]:
                continue
        visited_by[v].add(lab)
        prev_by_label[lab][v] = u
        q_rr.append((v, lab, d + 1))

bridges_rr = set().union(*bridge_paths_rr) if bridge_paths_rr else set()
info(f"[3h-bridge B] Ring<->Ring corridors: {len(bridge_paths_rr)} | band nodes={len(bridges_rr)}")


 #Merge additions (band corridors + HV1 endpoints) 
bridges_all = (bridges_hv1 | bridges_rr) - barrier_ids
# Do NOT include HV1 Dev in endpoints
hv1_endpoints_to_add -= hv1_dev_ids
# Augment the selection BEFORE building sel_df
sel_ids = set(sel_ids)
sel_ids |= bridges_all | hv1_endpoints_to_add
info(f"[3h-bridge] Added band corridors: {len(bridges_all)} | hv1_endpoints={len(hv1_endpoints_to_add)} "
     f"| final sel_ids now {len(sel_ids)}")

# 8) Build and save output
# sel_ids = set(visited.keys()) | set(seed_ids)
# sel_df = pd.DataFrame({grid_id_col: list(sel_ids)})
# sel_df["ring"] = sel_df[grid_id_col].map(visited).fillna(0).astype(int)
# out_3h = grid_full.merge(sel_df, on=grid_id_col, how="inner")
sel_df = pd.DataFrame({grid_id_col: list(sel_ids)})
# Preserve BFS rings; mark bridge cells distinctly
sel_df["ring"] = sel_df[grid_id_col].map(visited).fillna(0).astype(int)
if 'bridges_all' in locals() and len(bridges_all) > 0:
    sel_df.loc[sel_df[grid_id_col].isin(bridges_all), "ring"] = -1
out_3h = grid_full.merge(sel_df, on=grid_id_col, how="inner")

# QA flags
# out_3h["is_seed_hv1"]   = out_3h[grid_id_col].isin(seed_hv1_ids)
out_3h["is_seed_rec"]   = out_3h[grid_id_col].isin(seed_rec_ids)
out_3h["is_seed_func"]  = out_3h[grid_id_col].isin(seed_func_ids)
out_3h["in_3g"]         = out_3h[grid_id_col].isin(allowed_3g_ids)
out_3h["is_barrier"]    = out_3h[grid_id_col].isin(barrier_ids)
if zone_col:
    out_3h["is_hv1_development"] = out_3h[grid_id_col].isin(hv1_dev_ids)

out_path = os.path.join(output_dir, f"step_3h_growth_{str(max_rings)}rings.parquet")
out_3h.to_parquet(out_path, index=False)

info(f"Saved Step 3h → {out_path}")
info(f"Stats: seeds_start={len(seed_ids)} | expanded(visited)={len(visited)} | output_hexes={len(out_3h)}")

In [ ]:
#check code from here down, manually seems to produce faster results 

In [ ]:
#step 3i Merge the cells selected in Step 3 h) with both the ‘Recruitment Forest’ and
# ‘Old Forest with Functional Characteristics’ layer and dissolve. Entitled the remaining area ‘Secondary Reserves’. 
if "out_3h" not in globals():
    out_3h_path = os.path.join(output_dir, f"step_3h_growth_{str(max_rings)}rings.parquet")
    info(f"Loading step 3h from {out_3h_path}")
    out_3h = gpd.read_parquet(out_3h_path)
    
if 'rec_for' not in globals():
    info('loading rec for')
    rec_for=gpd.read_parquet(rec_for_path)
    
if 'func_for' not in globals():
    info('loading func for')
    func_for=gpd.read_parquet(os.path.join(output_dir, "step_3b_Old_Forest_with_Functional_Characteristics.parquet"))

if "ring" in out_3h.columns:
    selected_hexes = out_3h
else:
    selected_hexes = out_3h

info(f"Selected 3h hexes: {len(selected_hexes)} | Rec: {len(rec_for)} | Func: {len(func_for)}")

# Union of hexes
hex_union = None
if not selected_hexes.empty:
    hex_union = shapely.union_all([g for g in selected_hexes.geometry.values if g is not None])
rf_union = None
rf_parts = []
if not rec_for.empty:
    rf_parts.extend([g for g in rec_for.geometry.values if g is not None])
if not func_for.empty:
    rf_parts.extend([g for g in func_for.geometry.values if g is not None])

if rf_parts:
    rf_union = shapely.union_all(rf_parts)
    
geoms_to_merge = [g for g in (hex_union, rf_union) if g is not None and not g.is_empty]
if not geoms_to_merge:
    raise ValueError("[3i] No geometries found to merge. Check upstream inputs.")

merged_geom = shapely.union_all(geoms_to_merge)
try:
    merged_geom = make_valid(merged_geom)
except Exception:
    merged_geom = shapely.buffer(merged_geom, 0)

secondary_reserves = gpd.GeoDataFrame(
    {"NAME": ["Secondary Reserves"]},
    geometry=[merged_geom],
    crs=out_3h.crs,
)

secondary_reserves = fix(secondary_reserves)

out_parquet = os.path.join(output_dir, "step_3i_secondary_reserves.parquet")
secondary_reserves.to_parquet(out_parquet, index=False)

info(f"Saved Secondary Reserves → {out_parquet}")

In [ ]:
#step 3j 
if "step_3d_stands" not in globals():
    step_3e_path = os.path.join(output_dir, "step_3d_Old_Forest_without_Functional_Characteristics.parquet")
    info(f"Loading Step 3e stands from {step_3e_path}")
    step_3e_stands = gpd.read_parquet(step_3e_path)

if "secondary_reserves" not in globals():
    sr_path = os.path.join(output_dir, "step_3i_secondary_reserves.parquet")
    info(f"Loading Secondary Reserves from {sr_path}")
    secondary_reserves = gpd.read_parquet(sr_path)

sr_union = shapely.union_all([g for g in secondary_reserves.geometry.values if g is not None])


# 1)
#    Mode A: keep stands entirely OUTSIDE Secondary Reserves (no intersection)
#    Mode B: subtract overlap from stands and keep remaining parts (if any)
MODE = "A" 

if MODE == "A":
    info("Mode A: Selecting stands completely outside Secondary Reserves.")
    outside_mask = ~step_3e_stands.intersects(sr_union)
    aspat_deferred = step_3e_stands.loc[outside_mask].copy()

elif MODE == "B":
    info("Mode B: Subtracting Secondary Reserves from stands and keeping remainders.")
    diffs = shapely.difference(step_3e_stands.geometry.values, sr_union)
    aspat_deferred = step_3e_stands.copy()
    aspat_deferred["geometry"] = diffs
    aspat_deferred = aspat_deferred[aspat_deferred.geometry.notna() & ~aspat_deferred.geometry.is_empty].copy()
    aspat_deferred = fix(aspat_deferred)

else:
    raise ValueError("MODE must be 'A' or 'B'.")

# 2) Attribute + QA

aspat_deferred["DESIGNATION"] = "Aspatially Deferred Old Forest"

if aspat_deferred.crs and str(aspat_deferred.crs).endswith("3005"):
    aspat_deferred["area_ha"] = aspat_deferred.geometry.area / 10_000.0

info(f"Aspatially Deferred features: {len(aspat_deferred)}")

stand_id_col = None
for cand in ["STAND_ID", "stand_id", "UID", "uid", "POLY_ID", "poly_id"]:
    if cand in aspat_deferred.columns:
        stand_id_col = cand
        break

if stand_id_col:
    info(f"Distinct stands in output ({stand_id_col}): {aspat_deferred[stand_id_col].nunique()}")

out_parquet = os.path.join(output_dir, "step_3j_aspatial_deferred_old_forest.parquet")
aspat_deferred.to_parquet(out_parquet, index=False)


info(f"Saved Aspatially Deferred Old Forest → {out_parquet}")

# manual steps for secondary reserves aspatial deferred old forest and conservations lands 

## Conservation lands
- merge hv1 protections, secondary reserves and Aspatially Deferred Old Forest inside TSUs
- remove any areas that overlap hv1 development and critical infrastructures 

for each layer
- if not already dissolved, dissolve it for a 'clean' slate of attributes 
- create new attribute 'Designation' (text), populate with layer name (conservation lands, etc)
- union dissolved layer with wmb, tsu(split by wmb), forest recruitment, thlb layer and AFLB

- select polygons that have the designation column populated
- erase HV1 dz and critical infrastructure and electrification infrastructure from dataset, so there are none overlapping  
- export data set with the following attributes: designation, BASIN_ID, NAME, TRAPLINE_AREA_IDENTIFIER,  SIFA, SIFA_2, For_REP, Rec_Cat, for_rep_short, thlb_fact, afhlb_fact, area_ha, thlb_area_ha, aflb_ha
- recalculate area_ha, thlb_area_ha (thlb_fact*area_ha) and aflb_ha(aflb_fact*area_ha) 



In [ ]:
# #read in features with specific attributes 
# bar_diss=gpd.read_parquet(os.path.join(input_parquets,'barriers_dissolved.parquet'))
# wmb=gpd.read_parquet(os.path.join(input_parquets,'priority_WMB.parquet'), columns=['NAME','BASIN_ID','geometry'])
# tsu=gpd.read_parquet(os.path.join(input_parquets,'tsu_wmb.parquet'),columns=['TRAPLINE_AREA_IDENTIFIER','geometry'])
# thlb=gpd.read_parquet(os.path.join(input_parquets,'thlb_all.parquet'),columns=['thlb_fact','aflb_fact','area_ha','thlb_area_ha','aflb_ha','geometry'])
# rec_for=gpd.read_parquet(os.path.join(input_parquets,'rec_class_all.parquet'),columns=['SIFA','SIFA2','Rec_Cat','Rec_Cat_short'])

In [ ]:
# scndry_rsrvs=gpd.read_parquet(os.path.join(output_dir, "step_3i_secondary_reserves.parquet"))
# aspa_def=gpd.read_parquet(os.path.join(output_dir, "step_3j_aspatial_deferred_old_forest.parquet"))

In [ ]:

# def strtree_union_overlay(
#     features: gpd.GeoDataFrame,
#     barriers: gpd.GeoDataFrame | None,
#     wmb: gpd.GeoDataFrame,
#     tsu: gpd.GeoDataFrame,
#     thlb: gpd.GeoDataFrame,
#     rec_for: gpd.GeoDataFrame,
#     designation_text: str,
# ) -> gpd.GeoDataFrame:
#     """
#     STRtree-based pairwise UNION overlay.

#     - No dissolving during union with modifier layers
#     - Modifier attributes are preserved
#     - Returns ONE GeoDataFrame filtered by designation
#     """

#     # ------------------------------------------------------------
#     # 1. Ensure features is a single polygon (geometry dissolve ONLY)
#     # ------------------------------------------------------------
#     if len(features) > 1:
#         base_geom = union_all(features.geometry.values)
#     else:
#         base_geom = features.geometry.iloc[0]

#     # ------------------------------------------------------------
#     # 2. Erase barriers (geometry only)
#     # ------------------------------------------------------------
#     if barriers is not None and not barriers.empty:
#         barrier_geom = union_all(barriers.geometry.values)
#         base_geom = base_geom.difference(barrier_geom)

#     if base_geom.is_empty:
#         return gpd.GeoDataFrame(
#             columns=["designation", "geometry"],
#             crs=features.crs,
#         )

#     # ------------------------------------------------------------
#     # Helper: pairwise union via STRtree (NO dissolving)
#     # ------------------------------------------------------------
#     def _pairwise_union(base_geom, layer):
#         tree = STRtree(layer.geometry.values)
#         geom_index = {id(g): i for i, g in enumerate(layer.geometry.values)}

#         rows = []

#         for cand in tree.query(base_geom):
#             idx = geom_index[id(cand)]
#             row = layer.iloc[idx]

#             if not base_geom.intersects(cand):
#                 continue

#             out_row = row.copy()
#             out_row["geometry"] = base_geom.union(cand)
#             out_row["designation"] = designation_text

#             rows.append(out_row)

#         if not rows:
#             return None

#         return gpd.GeoDataFrame(rows, crs=layer.crs)

#     # ------------------------------------------------------------
#     # 3. Run pairwise union overlays
#     # ------------------------------------------------------------
#     outputs = []

#     for layer in (wmb, tsu, thlb, rec_for):
#         res = _pairwise_union(base_geom, layer)
#         if res is not None:
#             outputs.append(res)

#     if not outputs:
#         return gpd.GeoDataFrame(
#             columns=["designation", "geometry"],
#             crs=features.crs,
#         )

#     # ------------------------------------------------------------
#     # 4. Combine and filter by designation
#     # ------------------------------------------------------------
#     out = gpd.GeoDataFrame(
#         pd.concat(outputs, ignore_index=True),
#         crs=features.crs,
#     )

#     out = out[out["designation"] == designation_text].copy()

#     return out


In [ ]:
# secondary_reserves=strtree_union_overlay(scndry_rsrvs,bar_diss,wmb,tsu,thlb,rec_for,'Secondary Reserves')
# out_parquet = os.path.join(output_dir, "step_3i_secondary_reserves.parquet")
# secondary_reserves.to_parquet(out_parquet)
# # aspatially_deferred=strtree_union_overlay(aspa_def,bar_diss,wmb,tsu,thlb,rec_for,'Secondary Reserves')
# out_parquet = os.path.join(output_dir, "step_3j_aspatial_deferred_old_forest.parquet")
# aspatially_deferred.to_parquet(out_parquet)

In [ ]:

# # ============================================================
# # Merge func_for + no_func_for + rec_for → dissolve → intersect with TSU/WMB
# # Recalculate area in ha → join to original TSU/WMB tables
# # ============================================================

# # --- Ensure required layers are present (load if needed) ---
# if 'func_for' not in globals():
#     func_for = gpd.read_parquet(os.path.join(output_dir, "step_3b_Old_Forest_with_Functional_Characteristics.parquet"))
#     info("[3k] Loaded func_for")
# if 'no_func_for' not in globals():
#     no_func_for = gpd.read_parquet(os.path.join(output_dir, "step_3d_Old_Forest_without_Functional_Characteristics.parquet"))
#     info("[3k] Loaded no_func_for")
# if 'rec_for' not in globals():
#     raise RuntimeError("[3k] 'rec_for' is required for this step but is not in scope.")

# # --- Step 0: Reproject to EPSG:3005 for area math ---
# tsu = _to_3005(tsu)
# wmb = _to_3005(wmb)
# func_for = _to_3005(func_for)
# no_func_for = _to_3005(no_func_for)
# rec_for = _to_3005(rec_for)

# # --- Step 1: Merge (concat), explode, and keep polygon-only ---
# try:
#     func_for = func_for.explode(index_parts=False)
#     no_func_for = no_func_for.explode(index_parts=False)
#     rec_for = rec_for.explode(index_parts=False)
# except TypeError:
#     func_for = func_for.explode(ignore_index=False)
#     no_func_for = no_func_for.explode(ignore_index=False)
#     rec_for = rec_for.explode(ignore_index=False)

# func_for_poly    = func_for.loc[func_for.geometry.geom_type.isin(["Polygon","MultiPolygon"]), ["geometry"]].copy()
# no_func_for_poly = no_func_for.loc[no_func_for.geometry.geom_type.isin(["Polygon","MultiPolygon"]), ["geometry"]].copy()
# rec_for_poly     = rec_for.loc[rec_for.geometry.geom_type.isin(["Polygon","MultiPolygon"]), ["geometry"]].copy()

# combo_parts = pd.concat([func_for_poly, no_func_for_poly, rec_for_poly], ignore_index=True)
# info(f"[3k] Merge parts: func={len(func_for_poly)}, nofunc={len(no_func_for_poly)}, rec={len(rec_for_poly)}, merged={len(combo_parts)}")

# # --- Step 2: Dissolve the merged layer (UNION) ---
# if combo_parts.empty:
#     of_union = gpd.GeoDataFrame(columns=["geometry"], geometry="geometry", crs=3005)
#     info("[3k] UNION empty (no polygonal features).")
# else:
#     of_union = combo_parts.dissolve()  # union of all polygons
#     of_union = of_union.set_geometry("geometry")
#     of_union.crs = 3005
#     info("[3k] UNION dissolved into a single multipart polygon feature.")

# # --- Step 3: Intersect UNION with TSU and WMB ---
# info("[3k][TSU] Intersecting UNION with TSU")
# ix_tsu = gpd.overlay(tsu[[tsu_id_col, "geometry"]].dropna(subset=["geometry"]),
#                      of_union[["geometry"]].dropna(subset=["geometry"]),
#                      how="intersection", keep_geom_type=True)

# info("[3k][WMB] Intersecting UNION with WMB")
# ix_wmb = gpd.overlay(wmb[[wmb_id, "geometry"]].dropna(subset=["geometry"]),
#                      of_union[["geometry"]].dropna(subset=["geometry"]),
#                      how="intersection", keep_geom_type=True)

# # --- Step 4: Recalculate area in ha (EPSG:3005 → m² / 10,000) & group by unit ---
# if not ix_tsu.empty:
#     ix_tsu["union_oldforest_area_ha"] = ix_tsu.geometry.area / 10_000.0
#     tsu_union_ha = ix_tsu.groupby(tsu_id_col)["union_oldforest_area_ha"].sum()
# else:
#     tsu_union_ha = pd.Series(dtype="float64", name="union_oldforest_area_ha")

# if not ix_wmb.empty:
#     ix_wmb["union_oldforest_area_ha"] = ix_wmb.geometry.area / 10_000.0
#     wmb_union_ha = ix_wmb.groupby(wmb_id)["union_oldforest_area_ha"].sum()
# else:
#     wmb_union_ha = pd.Series(dtype="float64", name="union_oldforest_area_ha")

# # --- Step 5: Join totals back to original TSU and WMB tables ---
# tsu[tsu_id_col] = tsu[tsu_id_col].astype(str)
# wmb[wmb_id] = wmb[wmb_id].astype(str)

# # Reindex series on string IDs to avoid dtype mismatches
# tsu_union_ha.index = tsu_union_ha.index.astype(str)
# wmb_union_ha.index = wmb_union_ha.index.astype(str)

# tsu_metrics = pd.DataFrame({tsu_id_col: tsu[tsu_id_col].unique()}).merge(
#     tsu_union_ha.rename("union_oldforest_area_ha"),
#     left_on=tsu_id_col, right_index=True, how="left"
# ).fillna({"union_oldforest_area_ha": 0.0})

# wmb_metrics = pd.DataFrame({wmb_id: wmb[wmb_id].unique()}).merge(
#     wmb_union_ha.rename("union_oldforest_area_ha"),
#     left_on=wmb_id, right_index=True, how="left"
# ).fillna({"union_oldforest_area_ha": 0.0})

# tsu_out = tsu.merge(tsu_metrics, on=tsu_id_col, how="left")
# wmb_out = wmb.merge(wmb_metrics, on=wmb_id, how="left")

# # --- Optional: write outputs ---
# tsu_out_path = os.path.join(output_dir, "step_3k_union_recruitment_oldforest_tsu.parquet")
# wmb_out_path = os.path.join(output_dir, "step_3k_union_recruitment_oldforest_wmb.parquet")
# tsu_out.to_parquet(tsu_out_path)
# wmb_out.to_parquet(wmb_out_path)
# info(f"[3k] Wrote {tsu_out_path}")
# info(f"[3k] Wrote {wmb_out_path}")


In [ ]:
#manually creating the  sr and ad with rec info 

In [ ]:
# #3m Calculate time until the 25% of each for FOR_Rep total area, covered by Secondary Reserves and Aspatially Deferred Old Forest, is entirely comprised of stands >140. 

# AGE_THRESHOLD = 140
# BASE_YEAR = 2026

# if "ad_with_classes" not in globals():
#     ad_path_intermidate = os.path.join(output_dir, "step_3l_aspatial_deferred_with_rec_classes.parquet")
#     info(f"[3j] Loading Aspatially Deferred Old Forest from {ad_path_intermidate}")
#     ad_with_classes = gpd.read_parquet(ad_path_intermidate)

# if "sr_with_classes" not in globals():
#     sr_path_intermidate = os.path.join(output_dir, "step_3l_secondary_reserves_with_rec_classes.parquet")
#     info(f"[3i] Loading Secondary Reserves from {sr_path_intermidate}")
#     sr_with_classes = gpd.read_parquet(sr_path_intermidate)
    
# if 'targets_long' not in globals():
#     id_cols = ['NAME', 'Rec_Cat']
#     targets=pd.read_csv(brwmb_targets_csv)
#     short_cols = [c for c in targets.columns if c.isalpha() and c not in id_cols]
#     targets_long = (
#     targets.melt(
#         id_vars=id_cols,
#         value_vars=short_cols,
#         var_name='Rec_Cat_short',
#         value_name='target_ha'
#     )
#     .assign(
#         NAME=lambda d: d['NAME'].astype(str).str.strip(),
#         Rec_Cat=lambda d: pd.to_numeric(d['Rec_Cat'], errors='coerce').astype('Int64'),
#         Rec_Cat_short=lambda d: d['Rec_Cat_short'].astype(str).str.strip(),
#         target_ha=lambda d: pd.to_numeric(d['target_ha'], errors='coerce')
#     )
#     .dropna(subset=['Rec_Cat'])
#     )


# targets_long['target_ha_25_pct']=targets_long['target_ha']*0.25

# sr_old_stands = sr_with_classes.loc[sr_with_classes['SIFA_2'] >= 140].copy()
# ad_old_stands = ad_with_classes.loc[ad_with_classes['SIFA_2'] >= 140].copy()


# for df in (sr_old_stands, ad_old_stands):
#     df['aflb_fact'] = pd.to_numeric(df['aflb_fact'], errors='coerce')
#     df['area_ha_geom'] = df.geometry.area / 10_000.0  # metres² -> hectares
#     df['aflb_area']    = df['area_ha_geom'] * df['aflb_fact']


# sr_old_stands = sr_old_stands.loc[sr_old_stands['aflb_area'].notna() & (sr_old_stands['aflb_area'] > 0)].copy()
# ad_old_stands = ad_old_stands.loc[ad_old_stands['aflb_area'].notna() & (ad_old_stands['aflb_area'] > 0)].copy()

    
# pool = pd.concat([sr_old_stands, ad_old_stands], ignore_index=True)

# aflb_totals = (pool
#                .groupby(["Rec_Cat", "Rec_Cat_short"], dropna=False)["aflb_area"]
#                .sum()
#                .rename("accum_aflb_ha")
#                .reset_index())

# pool_poly = pool.loc[pool.geometry.geom_type.isin(["Polygon", "MultiPolygon"])].copy()


# targets = targets_long.copy()
# targets["Rec_Cat"] = pd.to_numeric(targets["Rec_Cat"], errors="coerce").astype("Int64")
# targets["Rec_Cat_short"] = targets["Rec_Cat_short"].astype(str).str.strip().str.upper()

# summary = (
#     targets
#     .merge(aflb_totals, on=["Rec_Cat", "Rec_Cat_short"], how="left")
#     .fillna({"accum_aflb_ha": 0.0})
# )

# summary['meets_25_pct_target'] = summary['accum_aflb_ha'] >= summary['target_ha_25_pct']

# mask_fail = ~summary['meets_25_pct_target']
# summary['shortfall_ha'] = pd.NA
# summary.loc[mask_fail, 'shortfall_ha'] = (
#     summary.loc[mask_fail, 'target_ha_25_pct'] - summary.loc[mask_fail, 'accum_aflb_ha']
# )

# # --- Build "younger stands" pool (SIFA_2 < 140) ---
# sr_yng_stands = sr_with_classes.loc[sr_with_classes['SIFA_2'] < 140].copy()
# ad_yng_stands = ad_with_classes.loc[ad_with_classes['SIFA_2'] < 140].copy()
# yng_pool = pd.concat([sr_yng_stands, ad_yng_stands], ignore_index=True)

# # Ensure key columns have the same normalization as `summary`
# yng_pool['Rec_Cat'] = pd.to_numeric(yng_pool['Rec_Cat'], errors='coerce').astype('Int64')
# yng_pool['Rec_Cat_short'] = yng_pool['Rec_Cat_short'].astype(str).str.strip().str.upper()

# yng_pool['aflb_fact'] = pd.to_numeric(yng_pool['aflb_fact'], errors='coerce')
# yng_pool['area_ha_geom'] = yng_pool.geometry.area / 10_000.0
# yng_pool['aflb_area'] = yng_pool['area_ha_geom'] * yng_pool['aflb_fact']


# yng_pool = yng_pool.loc[yng_pool['aflb_area'].notna() & (yng_pool['aflb_area'] > 0)].copy()


# yng_pool = yng_pool.merge(
#     summary[['Rec_Cat', 'Rec_Cat_short', 'shortfall_ha']],
#     on=['Rec_Cat', 'Rec_Cat_short'],
#     how='left'
# )

# yng_pool['shortfall_ha'] = pd.to_numeric(yng_pool['shortfall_ha'], errors='coerce')
# yng_pool = yng_pool.loc[yng_pool['shortfall_ha'].notna() & (yng_pool['shortfall_ha'] > 0)].copy()

# keys = ['Rec_Cat', 'Rec_Cat_short']
# yng_pool = yng_pool.sort_values(keys + ['SIFA_2'], ascending=[True, True, False]).reset_index(drop=True)

# yng_pool['cum_aflb_area'] = yng_pool.groupby(keys, observed=True)['aflb_area'].cumsum()
# yng_pool['prev_cum_aflb_area'] = yng_pool.groupby(keys, observed=True)['cum_aflb_area'].shift(fill_value=0)

# sel = (
#     (yng_pool['cum_aflb_area'] < yng_pool['shortfall_ha']) |
#     ((yng_pool['cum_aflb_area'] >= yng_pool['shortfall_ha']) &
#      (yng_pool['prev_cum_aflb_area'] < yng_pool['shortfall_ha']))
# )


# backfill_stands = yng_pool.loc[sel].copy()
# backfill_stands['selected_for_backfill'] = True

# age_cutoff = (
#     backfill_stands
#     .groupby(keys, observed=True)['SIFA_2']
#     .min()
#     .rename('youngest_selected_age')
# )
# added_by_group = (
#     backfill_stands
#     .groupby(keys, observed=True)['aflb_area']
#     .sum()
#     .rename('added_aflb_ha')
# )
# summary = (
#     summary
#     .merge(age_cutoff, on=keys, how='left')
#     .merge(added_by_group, on=keys, how='left')
# )



# backfill_stands.to_parquet(os.path.join(output_parquets,'step_3m_backfill_stands.parquet'))
# summary.to_excel(os.path.join(table_outputs,'step_3m_backfill_stands.xlsx'))